# M20a — Structural path comparison

**Author:** Ildefons Magrans de Abril  
**Affiliation:** Universitat Politècnica de Catalunya - BarcelonaTech (UPC)

**Purpose.** Compare allocation temperature with recurrent gain, leak rate, and hard top-k sparsity under the same calibration protocol.

**Provenance.** The seven original task names and the main experimental dimensions are recovered. The original notebook binary and exact original random seed were not recoverable. The executable run uses the fully preserved M23/M26 code lineage (`seed=20260718`) and is reported as a fresh protocol replication, not as a claim of byte-identical historical reproduction.

In [1]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import tcr_core as tcr

FROZEN = ROOT / "results" / "frozen"
REPRO = ROOT / "results" / "reproduced"
REPRO.mkdir(parents=True, exist_ok=True)

In [2]:
SEED=20260718
TRIALS=4
TASKS=["controlled_d1_clean","controlled_d20_clean","controlled_d20_white_plus_distractor","memory_d10","narma10","mackey_glass","lorenz_x"]
PATHS=["temperature","gain","leak","sparsity"]
CONFIG=dict(N=60,K=13,lengths=(1200,500,500),washout=100,input_scale=0.8,ridge=1e-5)
hist=pd.read_csv(FROZEN/"historical_reference_metrics.csv")
display(hist[hist.notebook_id=="M20a"])

,notebook_id,metric,value,ci_low,ci_high,provenance
18,M20a,temperature_support_range,55.19000,NaN,NaN,historical manuscript result
19,M20a,sparsity_support_range,58.00000,NaN,NaN,historical manuscript result
20,M20a,temperature_spectral_radius_range,0.04600,NaN,NaN,M21 audit of historical M20a
21,M20a,gain_spectral_radius_range,1.43800,NaN,NaN,M21 audit of historical M20a
22,M20a,temperature_safe_gain,0.00504,0.00263,NaN,historical manuscript result; only lower CI re...
23,M20a,temperature_safe_minus_shuffled_gain,0.02303,0.01170,NaN,historical manuscript result; only lower CI re...
24,M20a,gain_safe_gain,0.00879,NaN,NaN,historical manuscript result
25,M20a,leak_safe_gain,0.01249,NaN,NaN,historical manuscript result
26,M20a,sparsity_safe_gain,0.00046,NaN,NaN,historical manuscript result
27,M20a,best_indicator_correlation,0.73300,NaN,NaN,historical manuscript result


## Fresh path-comparison replication

In [3]:
start=time.time()
rep=tcr.run_panel(TASKS,TRIALS,PATHS,seed=SEED,**CONFIG)
rep.to_csv(REPRO/"m20a_replication_case_metrics.csv",index=False)
summary=(rep.groupby("path",as_index=False)
         .agg(n_cases=("task","size"),mean_safe_width=("safe_width","mean"),
              safe_gain=("safe_gain","mean"),safe_minus_matched_gain=("safe_minus_matched_gain","mean"),
              near_containment=("near_contained","mean"),exact_containment=("exact_contained","mean"),
              support_range=("support_range","mean"),spectral_radius_range=("spectral_radius_range","mean")))
summary.to_csv(REPRO/"m20a_replication_summary.csv",index=False)
display(summary.round(6))
print(f"elapsed: {time.time()-start:.2f} s")

,path,n_cases,mean_safe_width,safe_gain,safe_minus_matched_gain,near_containment,exact_containment,support_range,spectral_radius_range
0,gain,28,3.964286,0.004235,0.034531,0.857143,0.785714,0.00000,1.569810
1,leak,28,4.857143,0.004241,0.061486,0.964286,0.928571,0.00000,0.000000
2,sparsity,28,1.285714,0.001973,0.038468,0.750000,0.428571,58.00000,0.151554
3,temperature,28,2.892857,0.002533,0.027508,0.857143,0.678571,55.16451,0.171865


elapsed: 13.52 s


The fresh replication is deliberately not tuned to the historical table. The archived manuscript values remain in `results/frozen/table_path_ablation.csv`; differences are visible rather than hidden.